# Fine-tune PhoBERT for Vietnamese Expense Classification

**Model:** `vinai/phobert-base`  
**Task:** 10-class expense category classification  
**Runtime:** Google Colab (T4 GPU, ~15 min)  

## Steps
1. Upload `phobert_training_full.csv` and `phobert_eval.csv` to this session (cell Upload)
2. Run all cells
3. Model saved to `phobert-expense-v1/` — download it in the last cell
4. (Optional) Upload to Hugging Face Hub

In [ ]:
# ── 0. Install dependencies ────────────────────────────────────────────────
!pip install -q transformers datasets scikit-learn huggingface_hub

In [ ]:
# ── 0b. Upload training files ──────────────────────────────────────────────
# Run this cell to upload phobert_training_full.csv and phobert_eval.csv
from google.colab import files

print("Select phobert_training_full.csv and phobert_eval.csv (hold Ctrl to select both)")
uploaded = files.upload()
print("\nUploaded files:", list(uploaded.keys()))

In [ ]:
# ── 1. Config ──────────────────────────────────────────────────────────────
MODEL_NAME    = "vinai/phobert-base"
HF_REPO_ID    = "lamhoangphuc/phobert-expense-v1"   # only needed if uploading to HF Hub
MAX_LENGTH    = 128
BATCH_SIZE    = 32
EPOCHS        = 5
LR            = 2e-5
SEED          = 42
TRAIN_CSV     = "phobert_training_full.csv"
EVAL_CSV      = "phobert_eval.csv"
OUTPUT_DIR    = "phobert-expense-v1"

CATEGORIES = [
    "Ăn uống", "Đi lại", "Mua sắm", "Giải trí",
    "Hoá đơn tiện ích", "Sức khoẻ", "Giáo dục",
    "Du lịch", "Nhà cửa", "Khác",
]
LABEL2ID = {c: i for i, c in enumerate(CATEGORIES)}
ID2LABEL = {i: c for i, c in enumerate(CATEGORIES)}

print(f"Categories ({len(CATEGORIES)}):")
for i, c in enumerate(CATEGORIES):
    print(f"  {i}: {c}")

In [ ]:
# ── 2. Load and inspect data ───────────────────────────────────────────────
import pandas as pd
from collections import Counter

train_df = pd.read_csv(TRAIN_CSV)
eval_df  = pd.read_csv(EVAL_CSV)

print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")
print("\nTrain label distribution:")
for cat, n in sorted(Counter(train_df['label']).items(), key=lambda x: -x[1]):
    print(f"  {cat}: {n}")

# Filter out any rows with unknown labels
train_df = train_df[train_df['label'].isin(CATEGORIES)].reset_index(drop=True)
eval_df  = eval_df[eval_df['label'].isin(CATEGORIES)].reset_index(drop=True)
print(f"\nAfter filtering — Train: {len(train_df)} | Eval: {len(eval_df)}")

In [ ]:
# ── 3. Tokenize ────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReceiptDataset(Dataset):
    def __init__(self, df):
        self.texts  = df['text'].tolist()
        self.labels = [LABEL2ID[l] for l in df['label']]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = ReceiptDataset(train_df)
eval_ds  = ReceiptDataset(eval_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
eval_loader  = DataLoader(eval_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Eval batches: {len(eval_loader)}")

In [ ]:
# ── 4. Load model ──────────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cpu':
    print("WARNING: No GPU detected. Training will be very slow (~2–4 hours).")
    print("Go to Runtime → Change runtime type → T4 GPU")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── 5. Class weights (handle imbalance) ────────────────────────────────────
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels = [LABEL2ID[l] for l in train_df['label']]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CATEGORIES)),
    y=train_labels,
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:")
for cat, w in zip(CATEGORIES, class_weights):
    print(f"  {cat}: {w:.2f}")

In [ ]:
# ── 6. Training loop ───────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

best_f1 = 0.0
best_epoch = 0

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['label'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, labs)
            total_loss += loss.item()
            preds = out.logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labs.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), acc, f1

print(f"{'Epoch':>5} {'Train Loss':>11} {'Val Loss':>9} {'Acc':>7} {'Macro F1':>9}")
print("-" * 50)

torch.manual_seed(SEED)
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['label'].to(device)
        optimizer.zero_grad()
        out  = model(input_ids=ids, attention_mask=mask)
        loss = criterion(out.logits, labs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss, acc, f1 = evaluate(eval_loader)
    marker = " <- best" if f1 > best_f1 else ""
    print(f"{epoch:>5} {train_loss:>11.4f} {val_loss:>9.4f} {acc:>7.1%} {f1:>9.1%}{marker}")

    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nBest: epoch {best_epoch}, macro F1 = {best_f1:.1%}")

In [ ]:
# ── 7. Per-class evaluation on eval set ────────────────────────────────────
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification

# Load best checkpoint
best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in eval_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out  = best_model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(batch['label'].tolist())

print(classification_report(
    all_labels, all_preds,
    target_names=CATEGORIES,
    zero_division=0,
))

In [ ]:
# ── 8. Download model to local machine ────────────────────────────────────
# Zip the model folder and download it — place at models/phobert-expense-v1/ in your project
import shutil
from google.colab import files

zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f"Zipped: {zip_path}")
files.download(zip_path)
print("Download started. Extract to models/phobert-expense-v1/ in your project.")

In [ ]:
# ── 9. (Optional) Upload to Hugging Face Hub ──────────────────────────────
# Skip this cell if you don't have a HF account
from huggingface_hub import notebook_login
notebook_login()   # paste your HF write token

In [ ]:
# ── 9b. Push to Hub ────────────────────────────────────────────────────────
best_model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ── 10. Quick smoke test ───────────────────────────────────────────────────
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_cases = [
    ("Highlands Coffee | Ca phe sua, Banh mi",        "An uong"),
    ("Grab | Chuyen xe tu Q1 den Q7",                 "Di lai"),
    ("Pharmacity | Thuoc paracetamol, Vitamin C",     "Suc khoe"),
    ("EVN HCMC | Tien dien thang 3",                  "Hoa don tien ich"),
    ("ILA | Hoc phi khoa Tieng Anh thang 4",          "Giao duc"),
]

print(f"{'Text':<50} {'Predicted':<25} {'Match?'}")
print("-" * 85)
for text, expected_en in test_cases:
    pred = pipe(text)[0]['label']
    print(f"{text[:49]:<50} {pred:<25}")